# Which Content Pages Should Be Reviewed First?

## An honest FlyRank refresh-priority study

**Author:** Mehak Zahra  
**Lane:** Refresh / Content Opportunity Scoring  
**Primary metric:** Precision@50  
**Deployed paper:** https://mehkzhra.github.io/FlyRank-ML-Internship/

This notebook is the executable evidence index for the deployed research paper. It uses the public-safe 30,000-page starter slice; the separate ML-04 notebook documents the gated warehouse query workflow. Results support contemporaneous human review—not future prediction or causal refresh claims.

## Abstract

This study asks which measurable FlyRank content pages an editor should review first when review capacity is limited. I analyzed a public-safe 30,000-page internship slice and used the supplied 30-day comparison to define an observed-decline proxy. I compared a frozen low-CTR visibility rule with Logistic Regression and Random Forest on the same 6,163-page holdout containing seven entirely unseen client groups. The simple rule achieved Precision@50 of 0.78, compared with 0.74 for Logistic Regression and 0.60 for Random Forest; the negative modeling result kept the transparent rule as the operational queue. The output is a human-reviewed FlyRank content triage playbook, not a forecast of Google's algorithm or evidence that refreshing a page causes recovery.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
            if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists())
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
model = json.loads((ROOT/'work'/'outputs'/'model_metrics.json').read_text())
audit = json.loads((ROOT/'work'/'outputs'/'validation_audit_metrics.json').read_text())
playbook = json.loads((ROOT/'work'/'outputs'/'action_playbook_metrics.json').read_text())
print(f'Loaded {len(df):,} page rows and three committed metric receipts.')

Loaded 30,000 page rows and three committed metric receipts.


## Introduction / problem statement

FlyRank content teams can have thousands of measurable pages but limited editorial review time. The decision is: **which pages deserve review first, and which evidence-based route fits each page?** False positives waste editor time and may prompt unnecessary changes; false negatives leave opportunities unreviewed. The queue routes pages to metadata review, refresh review, content expansion, engagement review, protection, or monitoring—with a mandatory human approval gate.

## Data

The modeled artifact is the bundled `content_refresh_anonymized.csv`: one row per pseudonymized content page, 30,000 rows and 44 columns, with trailing-90-day aggregates and adjacent 30-day comparison windows. Repository documentation links it to warehouse release `flyrank_pseudonymized_warehouse_release_v20260703`; this model does **not** claim to be a full 78.8M-row warehouse benchmark.

Excluded: names, domains, URLs, titles, raw queries, credentials, IDs as features, `trend_direction`, `trend_pct`, and last/previous-30-day label components.

In [2]:
summary = pd.Series({
 'rows':len(df),'columns':df.shape[1],'unique_pages':df.content_id.nunique(),
 'client_groups':df.client_id.nunique(),
 'observed_decline_rows':int(df.trend_direction.eq('down').sum()),
 'observed_decline_rate':f"{df.trend_direction.eq('down').mean():.2%}"})
print(summary.to_string())
assert len(df) == df.content_id.nunique()
assert not {'client_name','domain','url','title','query'} & set(df.columns)

rows                      30000
columns                      44
unique_pages              30000
client_groups                32
observed_decline_rows     16262
observed_decline_rate    54.21%


## Methodology

The proxy is positive when supplied `trend_direction` is `down`: last-30-day impressions are more than 20% below the prior 30 days. The frozen rule multiplies visibility percentile, position-1-to-20 opportunity, a CTR gap below 0.50%, and a 100-impression gate. Logistic Regression and a constrained Random Forest use safe numeric/categorical source fields with imputation and encoding.

`GroupShuffleSplit` holds out seven entire clients (6,163 pages) from 25 training clients. Seed 42 is fixed. A deliberate `trend_pct` leak drove grouped AUC to 0.999, confirming the test could detect answer leakage; the field was removed. Trailing-90-day inputs still overlap the contemporaneous proxy windows, so this is triage rather than leakage-clean forecasting.

In [3]:
print(f"Train rows/clients: {model['train_rows']:,} / {model['train_clients']}")
print(f"Test rows/clients:  {model['test_rows']:,} / {model['test_clients']}")
print(f"Client overlap:      {audit['client_overlap_after']}")
print(pd.DataFrame(audit['leakage_test']).to_string(index=False))
assert audit['client_overlap_after'] == 0

Train rows/clients: 23,837 / 25
Test rows/clients:  6,163 / 7
Client overlap:      0
                     feature_set  roc_auc  precision_at_50
                 honest features    0.580             0.74
with trend_pct (deliberate leak)    0.999             1.00


## Results

All methods below use the same grouped test rows. The frozen rule identifies 39 proxy-positive pages among its first 50, Logistic Regression 37, and Random Forest 30. Logistic Regression improves Precision@10, and Random Forest has the strongest overall ROC AUC, but neither learned model beats the transparent rule at the chosen operational cutoff.

In [4]:
comparison = pd.DataFrame(model['comparison'])[
 ['method','base_rate','precision_at_10','precision_at_50','roc_auc',
  'average_precision','recall','f1']]
print(comparison.to_string(index=False))
winner = comparison.loc[comparison.precision_at_50.idxmax(),'method']
assert winner == 'Frozen rule baseline'
print(f'Operational winner at Precision@50: {winner}')

              method  base_rate  precision_at_10  precision_at_50  roc_auc  average_precision  recall    f1
Frozen rule baseline      0.511              0.7             0.78    0.562              0.561   0.033 0.063
 Logistic regression      0.511              0.8             0.74    0.580              0.577   0.633 0.596
       Random forest      0.511              0.6             0.60    0.612              0.596   0.611 0.602
Operational winner at Precision@50: Frozen rule baseline


## Limitations & honest framing

- The modeled artifact is the 30,000-page starter slice, not the complete daily warehouse.
- The target is a contemporaneous operational proxy, not future performance or refresh success.
- Trailing-90-day features overlap the proxy windows.
- One grouped holdout is not an untouched final confirmation set.
- Missing analytics can differ from genuine zero activity.
- Importance describes model behavior, not causation.
- Acting on a queue requires prospective follow-up to estimate impact.

## Ranked recommendations

1. Review the first 50 high-visibility, low-CTR candidates; privately verify intent, SERP features, tracking, seasonality, and business context.
2. Route by reason code to metadata, refresh, expansion, engagement, protection, or monitoring review.
3. Never auto-publish, prune, redirect, de-index, or promise recovery.
4. Record acceptance, action, time cost, and later outcomes.
5. Upgrade to non-overlapping historical features and future labels before forecasting claims.

In [5]:
actions = pd.Series(playbook['action_counts'],name='pages').rename_axis('action').sort_values(ascending=False)
print(actions.to_string())
print(f"Validated queue rows: {playbook['validated_queue_rows']:,}")
print(f"Automated content actions: {len(playbook['automated_actions'])}")
assert playbook['automated_actions'] == []

action
monitor                       3346
review_title_and_snippet      2278
refresh_facts_and_examples     239
protect_and_monitor            236
review_content_experience       64
Validated queue rows: 6,163
Automated content actions: 0


## Reproducibility

- [ML-07 baseline](https://github.com/mehkzhra/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb)
- [ML-08 modeling](https://github.com/mehkzhra/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb)
- [ML-09 validation audit](https://github.com/mehkzhra/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb)
- [ML-10 action playbook](https://github.com/mehkzhra/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb)
- [Repository](https://github.com/mehkzhra/FlyRank-ML-Internship)

Seeds and exact result tables live in committed JSON receipts under `work/outputs/`. Queue CSVs remain gitignored and are regenerated by their notebooks.

## Acknowledgments & data credit

Author: **Mehak Zahra**. Built on the [FlyRank ML Internship dataset](https://flyrank.ai/). I acknowledge the FlyRank internship team for the anonymized teaching data, documentation, and public-safety guidance.

## Self-check

- [x] All nine paper sections are represented.
- [x] Abstract has five sentences: question → data → method → result → intended use.
- [x] Model and baseline use the same grouped split and metric.
- [x] Leakage test, limitations, ranked recommendations, reproducibility, and data credit are explicit.
- [x] No private client information, raw queries, URLs, domains, or credentials are displayed.
- [x] All code cells are executed with visible outputs.